In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# No-op when you already have the thermo package alongside this notebook (the
# normal case: you cloned the repository and are running from code/chNN/).
# In Colab there is no repository, so fetch the package and the property data.
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------

# Homework: Temperature of ethane in a throttling process (package version)

> **Package version.** This notebook uses the shared `thermo` package (`PengRobinson`) instead of re-deriving the Peng–Robinson EOS inline. Compare with the self-contained companion [`PR_throttle_C2H6_homework.ipynb`](PR_throttle_C2H6_homework.ipynb), which builds every routine from scratch.
>
> Critical constants ($T_c$, $P_c$), the acentric factor $\omega$, and the ideal-gas $C_p$ coefficients are read from `code/data/pure_property.csv` via `PengRobinson.from_database`, so the numbers can differ slightly from the hand-entered SIS Table 6.6-1 values in the self-contained notebook.

## Problem statement

Ethane gas undergoes a rapid, adiabatic expansion through a continuous throttling process from upstream conditions $100^\circ\mathrm{C}$ and 100 bar to atmospheric pressure. Calculate the downstream gas temperature.

The throttle is isenthalpic: $\Delta\underline{H}^\mathrm{IG} + [\underline{H}-\underline{H}^\mathrm{IG}]_2 - [\underline{H}-\underline{H}^\mathrm{IG}]_1 = 0.$

## The Peng-Robinson EOS, from the `thermo` package

The generalized Peng-Robinson equation of state (SIS Eq. 6.4-2) is

$$ P = \frac{RT}{\underline{V}-b} - \frac{a(T)}{\underline{V}(\underline{V}+b) + b(\underline{V}-b)} \tag{Eq. 6.4-2}$$

with $b = 0.07780\,RT_c/P_c$ (Eq. 6.7-2), $a(T)=0.45724\,R^2T_c^2/P_c\,\alpha(T)$ (Eq. 6.7-1), $\sqrt{\alpha}=1+\kappa(1-\sqrt{T/T_c})$ (Eq. 6.7-3), and $\kappa = 0.37464 + 1.54226\omega - 0.26992\omega^2$ (Eq. 6.7-4).

All of this — building the cubic, taking its compressibility roots (SIS Table 6.4-3), and the departure functions below — is implemented in `thermo.PengRobinson`. The self-contained companion notebook codes each of these by hand; here we just call them.

The enthalpy and entropy departure functions for the PR EOS (SIS Eqs. 6.4-29, 6.4-30) are

$$[\underline{H} - \underline{H}^\mathrm{IG}] = RT(Z-1) + \frac{T\,da/dT-a}{2\sqrt2\,b}\ln\!\left[\frac{Z+(1+\sqrt2)B}{Z+(1-\sqrt2)B}\right] \tag{Eq. 6.4-29}$$

$$[\underline{S} - \underline{S}^\mathrm{IG}] = R\ln(Z-B) + \frac{da/dT}{2\sqrt2\,b}\ln\!\left[\frac{Z+(1+\sqrt2)B}{Z+(1-\sqrt2)B}\right] \tag{Eq. 6.4-30}$$

and are returned by `pr.departure_H(T, P, phase)` and `pr.departure_S(T, P, phase)` (J/mol and J/(mol·K)). The package uses the correct $R\ln(Z-B)$ in the entropy departure.

In [ ]:
import sys; sys.path.append("..")   # so `import thermo` works from the chapter folder
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants, optimize
from thermo import PengRobinson

R = constants.R

In [ ]:
pr = PengRobinson.from_database('ethane')
print(f"{pr.name}: Tc={pr.Tc:.1f} K, Pc={pr.Pc/1e5:.2f} bar, omega={pr.omega:.3f}")

ethane: Tc=305.4 K, Pc=48.80 bar, omega=0.099


In [ ]:
# Ideal-gas Cp correlation and its integrals, using the package coefficients pr.cp
def Cp_IG(pr, T):
    A, B, C, D = pr.cp
    return A + B*T + C*T**2 + D*T**3

def dH_IG(pr, T1, T2):
    A, B, C, D = pr.cp
    return (A*(T2-T1) + B*(T2**2-T1**2)/2
            + C*(T2**3-T1**3)/3 + D*(T2**4-T1**4)/4)

def dS_IG(pr, T1, T2, P1, P2):
    A, B, C, D = pr.cp
    return (A*np.log(T2/T1) + B*(T2-T1) + C*(T2**2-T1**2)/2
            + D*(T2**3-T1**3)/3 - R*np.log(P2/P1))

In [ ]:
# Known conditions
T1, P1 = 100 + 273.15, 100e5     # inlet: 100 C, 100 bar
P2      = 101325                 # outlet: 1 atm

Z1 = pr.Z(T1, P1, "vapor")
depH1 = pr.departure_H(T1, P1, "vapor")
print(f"Z1 = {Z1:.3f},  [H-H_IG]_1 = {depH1:.0f} J/mol")

Z1 = 0.607,  [H-H_IG]_1 = -5106 J/mol


## Guess an outlet temperature

The outlet temperature is unknown. Guess a $T_2$, compute $\Delta\underline{H}^\mathrm{IG}(T_1\to T_2)$ and the outlet departure $[\underline{H}-\underline{H}^\mathrm{IG}]_2(T_2,P_2)$, and check whether the isenthalpic sum is zero. Iterate on $T_2$ until it is.

In [ ]:
T2 = 285.0    # <-- your guess (K)

dHIG = dH_IG(pr, T1, T2)
depH2 = pr.departure_H(T2, P2, "vapor")
balance = dHIG + depH2 - depH1
print(f"T2 = {T2:.2f} K -> Delta H = {balance:.1f} J/mol (want 0)")

T2 = 285.00 K -> Delta H = 38.0 J/mol (want 0)


## (Solution) Let a root finder do the iteration

Once you understand the guess-and-check, you can solve for $T_2$ directly.

In [ ]:
def residual(T2):
    return dH_IG(pr, T1, T2) + pr.departure_H(T2, P2, "vapor") - depH1

T2_sol = optimize.brentq(residual, 200.0, T1)
print(f"Outlet temperature T2 = {T2_sol:.2f} K  ({T2_sol-273.15:.2f} C)")

Outlet temperature T2 = 284.26 K  (11.11 C)
